# Load simulated data

In [ ]:
import datetime

import numpy as np
from stonesoup.types.state import GaussianState
from stonesoup.models.measurement.linear import LinearGaussian

from theia.coordinates import CoordinateTransformations
from theia.simulation.logging import LogLoader

# How to go from the 6D state space (positions + velocities) to the
# 3D measurement space (positions only).
measurement_model = LinearGaussian(
    ndim_state=6,
    mapping=(0, 2, 4),
    noise_covar=np.identity(3),
)

loader = LogLoader("log_opensky.json", measurement_model)

# Select a single radar and a single target

In [ ]:
RADAR_ID = 0
TARGET_ID = 7

detections = [
    d
    for d in loader.blue_monostatic_radar_detections
    if d.metadata["radar_id"] == RADAR_ID and d.metadata["target_id"] == TARGET_ID
]

radar = next(
    radar for radar in loader.blue_monostatic_radars if radar.receiver.id == RADAR_ID
)
ground_truth = loader.red_target_ground_truth[TARGET_ID]

# Build tracker

In [ ]:
from stonesoup.models.transition.linear import (
    CombinedLinearGaussianTransitionModel,
    ConstantVelocity,
)
from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.updater.kalman import KalmanUpdater

q = 0.05

transition_model = CombinedLinearGaussianTransitionModel(
    [ConstantVelocity(q), ConstantVelocity(q), ConstantVelocity(q)]
)

predictor = KalmanPredictor(transition_model)
updater = KalmanUpdater(measurement_model)

# Prior: Zero velocity, position of first detection.
first_detection = sorted(
    [d for d in detections if d.metadata["target_id"] == TARGET_ID],
    key=lambda d: d.timestamp,
)[0]

prior = GaussianState(
    [
        [first_detection.state_vector[0]],
        [0],
        [first_detection.state_vector[1]],
        [0],
        [first_detection.state_vector[2]],
        [0],
    ],
    covar=np.identity(6),
    timestamp=first_detection.timestamp,
)

# Track

In [ ]:
from stonesoup.types.track import Track
from stonesoup.types.hypothesis import SingleHypothesis

track = Track()
for detection in detections:
    prediction = predictor.predict(prior, timestamp=detection.timestamp)
    hypothesis = SingleHypothesis(prediction, detection)
    post = updater.update(hypothesis)
    track.append(post)
    prior = post

# Plot ground truth and detections

In [ ]:
times = [state.timestamp for state in ground_truth]

In [ ]:
from stonesoup.plotter import AnimatedPlotterly

plotter = AnimatedPlotterly(times[::5])
# Order: [x, vx, y, vy, z, vz] -> Need indices 0, 2, 4
plotter.plot_ground_truths(ground_truth, (0, 2))
plotter.plot_measurements(detections, (0, 2))
plotter.plot_tracks(track, (0, 2), uncertainty=True)
plotter.fig

In [ ]:
from typing import Iterable
import shapely
from stonesoup.types.state import State


def stonesoup_states_to_shapely(states: Iterable[State]) -> shapely.LineString:
    coordinates = []
    for state in states:
        lat, lon, alt = CoordinateTransformations.cartesian_to_geodetic(
            state.state_vector[0],
            state.state_vector[2],
            state.state_vector[4],
        )
        coordinates.append((lon, lat, alt))
    return shapely.LineString(coordinates)

In [ ]:
true_trajectory_xyz = np.stack(
    [state.state_vector.flatten() for state in ground_truth.states]
)
true_trajectory_times = [state.timestamp for state in ground_truth.states]
tracked_trajectory_xyz = np.stack(
    [state.state_vector.flatten() for state in track.states]
)
tracked_trajectory_times = [state.timestamp for state in track.states]

# diff = true_trajectory - tracked_trajectory

In [ ]:
from matplotlib import pyplot as plt


fig, axes = plt.subplots(ncols=3)

t0 = true_trajectory_times[0]

for i in range(3):
    ax = axes[i]
    ax.plot(true_trajectory_times, true_trajectory_xyz[:, i * 2])
    ax.plot(tracked_trajectory_times, tracked_trajectory_xyz[:, i * 2], "--")

    for detection in detections:
        ax.axvline(detection.timestamp, color="red", linewidth=0.5)
    ax.tick_params(axis="x", rotation=90)
    ax.set_xlim(
        [t0 + datetime.timedelta(minutes=5), t0 + datetime.timedelta(minutes=13)]
    )

fig.tight_layout()

In [ ]:
ax.get_xlim()[1] - ax.get_xlim()[0]

In [ ]:
from theia.distance import get_2d_distance_between_locs_heights

ground_truth_lonlatalt = stonesoup_states_to_shapely(ground_truth.states).coords

latlon_of_interest = (47.4872, 8.9635)
diffs = [
    get_2d_distance_between_locs_heights(
        latlon_of_interest[0],
        latlon_of_interest[1],
        0,
        c[1],
        c[0],
        0,
    )
    for c in ground_truth_lonlatalt
]

i = int(np.argmin(diffs))
alt = ground_truth_lonlatalt[i][2]
alt

In [ ]:
from theia.coverage import calculate_coverage
from theia.radar_equation import calculate_maximum_monostatic_range

RCS = 2.0

coverage = calculate_coverage(
    radar.receiver.point,
    calculate_maximum_monostatic_range(radar, RCS),
    alt,
)

In [ ]:
from theia.mapping import RadarMap

mapper = RadarMap(
    radars={"radar": radar},
    paths={
        "ground truth": stonesoup_states_to_shapely(ground_truth.states),
        "tracked": stonesoup_states_to_shapely(track.states),
    },
    polygons={"coverage": coverage},
)
mapper.to_map()